In [1]:
from qutip import *
import numpy as np
from contextuality.measurement_scenario import MeasurementScenarioImplementations
from contextuality.empirical_model import EmpiricalModel

In [2]:
KCBS = MeasurementScenarioImplementations.KCBS()

angle = np.pi / 5
Z_angle = np.sqrt(np.cos(angle))

zero, one, two = [basis(3, i) for i in range(3)]

# In Simple Hardy-Like Proof... By Cabello

ket1 = zero + Z_angle * two
ket1 = ket1.unit()

ket2 = np.cos(4*angle) * zero + np.sin(4*angle) * one + Z_angle * two
ket2 = ket2.unit()

ket3 = np.cos(2*angle) * zero - np.sin(2*angle) * one + Z_angle * two
ket3 = ket3.unit()

ket4 = np.cos(2*angle) * zero + np.sin(2*angle) * one + Z_angle * two
ket4 = ket4.unit()

ket5 = np.cos(4*angle) * zero - np.sin(4*angle) * one + Z_angle * two
ket5 = ket5.unit()

# kets = [ (np.cos(4*i*angle) * zero + np.sin(4*i*angle) * one + np.cos(angle) * two).unit() for i in range(5)]

Ps = [eval(f"ket2dm(ket{i})") for i in range(1,6)]
# Ps = [ket2dm(k) for k in kets]
As = [identity(3) - 2*P for P in Ps]
PVMs = [ [ket2dm(a.eigenstates()[1][0]).full(), (ket2dm(a.eigenstates()[1][1]) + ket2dm(a.eigenstates()[1][2])).full()] for a in As]

In [3]:
state = ket2dm(two).unit()
PVMs2 = [ [ket2dm(a.eigenstates()[1][0]), (ket2dm(a.eigenstates()[1][1]) + ket2dm(a.eigenstates()[1][2]))] for a in As]
# p_eq = [ (PVMs2[i][0] * PVMs2[(i+1) % 5][0] * state).tr() + (PVMs2[i][1] * PVMs2[(i+1) % 5][1] * state).tr() for i in range(5)]
p_eq = [ (Ps[i] * Ps[(i+1) % 5] * state).tr() + ((identity(3) - Ps[i]) * (identity(3) - Ps[(i+1) % 5]) * state).tr() for i in range(5)]
p_eq2 = [ ((Ps[i] * Ps[(i+1) % 5] * state).tr(), ((identity(3) - Ps[i]) * (identity(3) - Ps[(i+1) % 5]) * state).tr()) for i in range(5)]
print(p_eq2, sum(p_eq))

[(6.180861293293861e-17, (0.10557280900008427+0j)), (6.297461701680791e-18, (0.10557280900008421+0j)), (-2.1458113913948123e-17, (0.10557280900008419+0j)), (6.297461701680791e-18, (0.10557280900008421+0j)), (6.180861293293861e-17, (0.10557280900008427+0j))] (0.5278640450004213+0j)


In [4]:
em = EmpiricalModel(KCBS)
rho = ket2dm(two).unit().full()
em.quantum_realisation(rho, PVMs)

In [5]:
print(em)
NCF = em.compute_NCF()['NCF']
print(f"The NCF is {NCF:.2f}")

EmpiricalModel(MeasurementScenario(X=[0, 1, 2, 3, 4], M=[[0, 1], [1, 2], [2, 3], [3, 4], [4, 0]], O=[0, 1])
	-0.00 0.45 0.45 0.11 
	0.00 0.45 0.45 0.11 
	0.00 0.45 0.45 0.11 
	0.00 0.45 0.45 0.11 
	-0.00 0.45 0.45 0.11 
)
The NCF is 0.53


In [6]:
# Quantum violation value:
CF = em.compute_NCF()['CF']
CF * -5 + (1-CF) * -3

np.float64(-3.9442719099991574)